In [5]:
"""
CS3807 Deep Learning Lab - Experiment 6 (Kaggle, T4, final)
Single-GPU version. All device scoping removed - TF uses the only visible GPU.
"""

import os
# Force a single GPU BEFORE importing TensorFlow. Eliminates all cross-device issues.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import glob
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, ConfusionMatrixDisplay,
)

np.random.seed(42)
tf.random.set_seed(42)

# ---- GPU check ----
gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs visible to TensorFlow: {len(gpus)}")
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass

# No mixed precision, no XLA, no MirroredStrategy - all three caused problems.
HAR_BATCH = 256
VIDEO_BATCH = 32
SEQ2SEQ_BATCH = 128
PLOT_DIR = "/kaggle/working"


# =============================
# Part A - HAR sequence classification
# =============================

HAR_URLS = [
    "https://github.com/MaxBenChrist/human-activity-dataset/blob/master/UCI%20HAR%20Dataset.zip?raw=True",
    "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip",
]
DATA_DIR = "/kaggle/working/har_data"

SIGNALS = ["body_acc_x", "body_acc_y", "body_acc_z",
           "body_gyro_x", "body_gyro_y", "body_gyro_z",
           "total_acc_x", "total_acc_y", "total_acc_z"]

ACTIVITY_NAMES = ["WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS",
                  "SITTING", "STANDING", "LAYING"]


def find_har_root(base):
    for path in glob.glob(os.path.join(base, "**", "Inertial Signals"), recursive=True):
        if os.path.basename(os.path.dirname(path)) == "train":
            return os.path.dirname(os.path.dirname(path))
    return None


def extract_all_nested(zip_path, dest):
    import zipfile
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)
    for nested_zip in glob.glob(os.path.join(dest, "**", "*.zip"), recursive=True):
        if nested_zip != zip_path:
            extract_all_nested(nested_zip, os.path.dirname(nested_zip))


def _progress(count, block_size, total_size):
    if total_size > 0:
        done = count * block_size
        pct = min(done / total_size * 100, 100)
        print(f"\rdownloading... {pct:5.1f}% ({done // (1024*1024)} MB)", end="", flush=True)


def download_har():
    import urllib.request
    os.makedirs(DATA_DIR, exist_ok=True)
    root = find_har_root(DATA_DIR)
    if root:
        return root

    zip_path = os.path.join(DATA_DIR, "har.zip")
    if not os.path.exists(zip_path):
        last_error = None
        for url in HAR_URLS:
            try:
                print(f"downloading HAR dataset from {url}")
                urllib.request.urlretrieve(url, zip_path, reporthook=_progress)
                print()
                break
            except Exception as e:
                print(f"\nthat source failed ({e}), trying the next one")
                last_error = e
                if os.path.exists(zip_path):
                    os.remove(zip_path)
        else:
            raise RuntimeError(f"couldn't download HAR dataset: {last_error}")

    extract_all_nested(zip_path, DATA_DIR)
    root = find_har_root(DATA_DIR)
    if root is None:
        raise RuntimeError("couldn't find 'Inertial Signals' folder after extracting")
    return root


def load_signals(root, split):
    folder = os.path.join(root, split, "Inertial Signals")
    channels = [np.loadtxt(os.path.join(folder, f"{sig}_{split}.txt")) for sig in SIGNALS]
    return np.stack(channels, axis=-1)


def load_labels(root, split):
    y = np.loadtxt(os.path.join(root, split, f"y_{split}.txt")).astype(int)
    return y - 1


def build_har_dataset(subset_size=3000):
    root = download_har()
    X_train_full = load_signals(root, "train")
    y_train_full = load_labels(root, "train")
    X_test = load_signals(root, "test")
    y_test = load_labels(root, "test")

    if subset_size and subset_size < len(X_train_full):
        per_class = subset_size // 6
        idx = []
        for c in range(6):
            class_idx = np.where(y_train_full == c)[0]
            idx.extend(class_idx[:per_class])
        idx = np.array(idx)
        np.random.shuffle(idx)
        X_train_full, y_train_full = X_train_full[idx], y_train_full[idx]

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=15 / 85,
        stratify=y_train_full, random_state=42
    )

    n_features = X_train.shape[-1]
    mean = X_train.reshape(-1, n_features).mean(axis=0)
    std = X_train.reshape(-1, n_features).std(axis=0) + 1e-8
    normalize = lambda x: (x - mean) / std
    X_train, X_val, X_test = normalize(X_train), normalize(X_val), normalize(X_test)

    print("Input tensor shape:")
    print(f"Training   : {X_train.shape}")
    print(f"Validation : {X_val.shape}")
    print(f"Testing    : {X_test.shape}")
    print(f"Number of classes: {len(ACTIVITY_NAMES)}")
    print(f"Number of features per time step: {n_features}")
    print(f"Sequence length: {X_train.shape[1]}")
    print(f"Class distribution (train): {np.bincount(y_train, minlength=6)}")

    return X_train, X_val, X_test, y_train, y_val, y_test


def plot_sample_signals(X, y):
    classes_to_show = [0, 3, 5]
    fig, axes = plt.subplots(len(classes_to_show), 1, figsize=(9, 8), sharex=True)
    for ax, c in zip(axes, classes_to_show):
        window = X[np.where(y == c)[0][0]]
        for ch, label in zip([0, 3, 6], ["body_acc_x", "body_gyro_x", "total_acc_x"]):
            ax.plot(window[:, ch], label=label)
        ax.set_title(ACTIVITY_NAMES[c])
        ax.legend(loc="upper right", fontsize=8)
    axes[-1].set_xlabel("Time step")
    fig.tight_layout()
    fig.savefig(f"{PLOT_DIR}/plot1_sample_signals.png", dpi=150)
    plt.close(fig)


def build_har_model(recurrent_type, timesteps, n_features, n_classes=6, units=32):
    layer_map = {"RNN": layers.SimpleRNN, "LSTM": layers.LSTM, "GRU": layers.GRU}
    rnn_kwargs = dict(units=units, activation="tanh", unroll=False, use_bias=True)
    if recurrent_type in ("LSTM", "GRU"):
        rnn_kwargs["recurrent_activation"] = "sigmoid"

    model = models.Sequential([
        layers.Input(shape=(timesteps, n_features)),
        layer_map[recurrent_type](**rnn_kwargs),
        layers.Dropout(0.2),
        layers.Dense(16, activation="relu"),
        layers.Dense(n_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def train_and_evaluate(name, X_train, y_train, X_val, y_val, X_test, y_test,
                       epochs=30, save_plots=True):
    model = build_har_model(name, X_train.shape[1], X_train.shape[2])

    train_ds = (
        tf.data.Dataset.from_tensor_slices((X_train, y_train))
        .shuffle(min(len(X_train), 10000), seed=42)
        .batch(HAR_BATCH, drop_remainder=True)
        .prefetch(tf.data.AUTOTUNE)
    )
    val_ds = (
        tf.data.Dataset.from_tensor_slices((X_val, y_val))
        .batch(HAR_BATCH).prefetch(tf.data.AUTOTUNE)
    )

    start = time.time()
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=0)
    train_time = time.time() - start

    if save_plots:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].plot(history.history["loss"], label="train")
        axes[0].plot(history.history["val_loss"], label="val")
        axes[0].set_title(f"{name} loss")
        axes[0].set_xlabel("epoch")
        axes[0].set_ylabel("loss")
        axes[0].legend()
        axes[1].plot(np.array(history.history["accuracy"]) * 100, label="train")
        axes[1].plot(np.array(history.history["val_accuracy"]) * 100, label="val")
        axes[1].set_title(f"{name} accuracy")
        axes[1].set_xlabel("epoch")
        axes[1].set_ylabel("accuracy (%)")
        axes[1].legend()
        fig.tight_layout()
        fig.savefig(f"{PLOT_DIR}/plot23_{name.lower()}_curves.png", dpi=150)
        plt.close(fig)

    y_pred = np.argmax(model.predict(X_test, batch_size=HAR_BATCH, verbose=0), axis=1)
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    if save_plots:
        fig, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay(cm, display_labels=ACTIVITY_NAMES).plot(
            ax=ax, xticks_rotation=45, colorbar=False)
        ax.set_title(f"{name} confusion matrix")
        fig.tight_layout()
        fig.savefig(f"{PLOT_DIR}/plot4_{name.lower()}_confusion.png", dpi=150)
        plt.close(fig)

    return {
        "model": name, "accuracy": acc * 100, "precision": prec * 100,
        "recall": rec * 100, "f1": f1 * 100, "params": model.count_params(),
        "train_time": train_time, "confusion": cm,
    }


def sequence_length_experiment(X_train, y_train, X_val, y_val, X_test, y_test,
                               lengths=(32, 64, 128)):
    f1_by_model = {name: [] for name in ["RNN", "LSTM", "GRU"]}
    for T in lengths:
        Xtr, Xv, Xte = X_train[:, :T, :], X_val[:, :T, :], X_test[:, :T, :]
        for name in f1_by_model:
            res = train_and_evaluate(name, Xtr, y_train, Xv, y_val, Xte, y_test,
                                     epochs=15, save_plots=False)
            f1_by_model[name].append(res["f1"])

    fig, ax = plt.subplots(figsize=(6, 4))
    for name, vals in f1_by_model.items():
        ax.plot(lengths, vals, marker="o", label=name)
    ax.set_xlabel("Sequence length (T)")
    ax.set_ylabel("Test macro F1 (%)")
    ax.set_title("Effect of sequence length")
    ax.legend()
    fig.tight_layout()
    fig.savefig(f"{PLOT_DIR}/plot6_sequence_length.png", dpi=150)
    plt.close(fig)
    return f1_by_model


def bptt_numerical_check():
    x = [0.5, 0.7, 0.2]
    Wx, Wh, b = 0.5, 0.8, 0.1
    h = 0.0
    hidden_states = []
    for xi in x:
        h = np.tanh(Wx * xi + Wh * h + b)
        hidden_states.append(h)
    print("h1, h2, h3 =", hidden_states)
    return hidden_states


# =============================
# Part B - video understanding
# =============================

N_FRAMES = 10
FRAME_SIZE = 224
VIDEOS_PER_CLASS = 40

UCF_ROOT_CANDIDATES = [
    "/kaggle/input/ucf101-action-recognition",
    "/kaggle/input/datasets/matthewjansen/ucf101-action-recognition",
    "/kaggle/input/matthewjansen/ucf101-action-recognition",
]

PREFERRED_CLASSES = ["Basketball", "Biking", "WalkingWithDog",
                     "TennisSwing", "JumpRope", "HorseRace", "Skiing"]


def find_ucf_root():
    for candidate in UCF_ROOT_CANDIDATES:
        if os.path.isdir(candidate) and all(
                os.path.isdir(os.path.join(candidate, s)) for s in ("train", "val", "test")):
            return candidate
    for candidate in glob.glob("/kaggle/input/**/", recursive=True):
        candidate = candidate.rstrip("/")
        if all(os.path.isdir(os.path.join(candidate, s)) for s in ("train", "val", "test")):
            return candidate
    return None


def pick_classes(train_dir, n=5):
    available = sorted(d for d in os.listdir(train_dir)
                       if os.path.isdir(os.path.join(train_dir, d)))
    chosen = [c for c in PREFERRED_CLASSES if c in available][:n]
    if len(chosen) < 3:
        chosen = available[:n]
    print(f"classes found under {train_dir}: {len(available)} total")
    print(f"classes selected for this run: {chosen}")
    return chosen


def sample_frames(video_path, n_frames=N_FRAMES, size=FRAME_SIZE):
    import cv2
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return np.array([])
    pick = set(np.linspace(0, max(total - 1, 0), n_frames).astype(int))
    frames = []
    i = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if i in pick:
            frame = cv2.resize(frame, (size, size))
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        i += 1
    cap.release()
    while frames and len(frames) < n_frames:
        frames.append(frames[-1])
    return np.array(frames[:n_frames])


def build_feature_extractor():
    # No tf.device wrapper - single GPU is visible, TF places it there automatically
    base = tf.keras.applications.MobileNetV2(
        include_top=False, weights="imagenet", pooling="avg",
        input_shape=(FRAME_SIZE, FRAME_SIZE, 3))
    base.trainable = False
    return base


def load_split(split_dir, classes, videos_per_class):
    paths, labels = [], []
    for label, cls in enumerate(classes):
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            print(f"warning: '{cls}' not found under {split_dir}, skipping")
            continue
        clips = sorted(os.listdir(cls_dir))[:videos_per_class]
        for fname in clips:
            paths.append(os.path.join(cls_dir, fname))
            labels.append(label)

    sample_for_plot = None
    if not paths:
        return np.array([]), np.array([]), sample_for_plot

    with ThreadPoolExecutor(max_workers=8) as ex:
        results = list(ex.map(sample_frames, paths))

    frames_list, good_labels = [], []
    for frames, label in zip(results, labels):
        if frames is None or len(frames) == 0:
            continue
        if sample_for_plot is None:
            sample_for_plot = (frames, classes[label])
        frames_list.append(frames)
        good_labels.append(label)

    return np.array(frames_list), np.array(good_labels), sample_for_plot


def extract_features_for_split(feature_extractor, frames):
    preprocess = tf.keras.applications.mobilenet_v2.preprocess_input
    n_videos = frames.shape[0]
    flat = frames.reshape(-1, FRAME_SIZE, FRAME_SIZE, 3).astype("float32")
    flat = preprocess(flat)
    feats = feature_extractor.predict(flat, batch_size=64, verbose=1)
    return feats.reshape(n_videos, N_FRAMES, -1)


def plot_frame_grid(frames, cls_name):
    fig, axes = plt.subplots(1, len(frames), figsize=(15, 2))
    for ax, frame in zip(axes, frames):
        ax.imshow(frame.astype("uint8"))
        ax.axis("off")
    fig.suptitle(f"sampled frames - {cls_name}")
    fig.savefig(f"{PLOT_DIR}/plot7_video_frames.png", dpi=150)
    plt.close(fig)


def build_video_classifier(recurrent_type, feature_dim, n_classes, units=32):
    layer_map = {"LSTM": layers.LSTM, "GRU": layers.GRU}
    model = models.Sequential([
        layers.Input(shape=(N_FRAMES, feature_dim)),
        layer_map[recurrent_type](
            units, activation="tanh", recurrent_activation="sigmoid"),
        layers.Dense(n_classes, activation="softmax"),
    ])
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model


def run_video_pipeline(recurrent_type="LSTM"):
    ucf_root = find_ucf_root()
    if ucf_root is None:
        print("couldn't find UCF101 dataset under /kaggle/input. "
              "Attach matthewjansen/ucf101-action-recognition and rerun.")
        return None
    print("using UCF101 data from:", ucf_root)

    classes = pick_classes(os.path.join(ucf_root, "train"))

    frames_train, y_train, sample = load_split(
        os.path.join(ucf_root, "train"), classes, VIDEOS_PER_CLASS)
    frames_val, y_val, _ = load_split(
        os.path.join(ucf_root, "val"), classes, VIDEOS_PER_CLASS)
    frames_test, y_test, _ = load_split(
        os.path.join(ucf_root, "test"), classes, VIDEOS_PER_CLASS)

    if sample is not None:
        plot_frame_grid(*sample)

    extractor = build_feature_extractor()
    print("extracting MobileNetV2 features for train ...")
    X_train = extract_features_for_split(extractor, frames_train)
    print("extracting MobileNetV2 features for val ...")
    X_val = extract_features_for_split(extractor, frames_val)
    print("extracting MobileNetV2 features for test ...")
    X_test = extract_features_for_split(extractor, frames_test)

    print("CNN feature dimension:", X_train.shape[-1])
    print("Tensor shape into the recurrent network (train):", X_train.shape)

    model = build_video_classifier(recurrent_type, X_train.shape[-1], len(classes))
    history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                        epochs=20, batch_size=VIDEO_BATCH, verbose=0)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(history.history["loss"], label="train")
    axes[0].plot(history.history["val_loss"], label="val")
    axes[0].set_title("video model loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()
    axes[1].plot(np.array(history.history["accuracy"]) * 100, label="train")
    axes[1].plot(np.array(history.history["val_accuracy"]) * 100, label="val")
    axes[1].set_title("video model accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("accuracy (%)")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(f"{PLOT_DIR}/plot8_video_curves.png", dpi=150)
    plt.close(fig)

    probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    print(f"video model test accuracy: {acc * 100:.2f}%")

    print("\nSample predictions (Predicted class / Actual class / Confidence):")
    for i in range(min(5, len(y_test))):
        pred_cls = classes[y_pred[i]]
        true_cls = classes[y_test[i]]
        conf = float(probs[i, y_pred[i]])
        print(f"  Predicted class: {pred_cls:15s} | "
              f"Actual class: {true_cls:15s} | Confidence: {conf:.4f}")

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=classes).plot(
        ax=ax, xticks_rotation=45, colorbar=False)
    fig.tight_layout()
    fig.savefig(f"{PLOT_DIR}/plot9_video_confusion.png", dpi=150)
    plt.close(fig)

    return model, history, cm


# =============================
# Part C - seq2seq reversal task
# =============================

def make_seq2seq_data(n_samples=5000, seq_len=6, vocab_size=10):
    X = np.random.randint(0, vocab_size, size=(n_samples, seq_len))
    y = X[:, ::-1]
    return X, y


def build_seq2seq_model(seq_len, vocab_size, latent_dim=64):
    encoder_inputs = layers.Input(shape=(seq_len,))
    enc_emb = layers.Embedding(vocab_size, latent_dim)(encoder_inputs)
    _, state_h, state_c = layers.LSTM(latent_dim, return_state=True)(enc_emb)

    decoder_inputs = layers.Input(shape=(seq_len,))
    dec_emb = layers.Embedding(vocab_size, latent_dim)(decoder_inputs)
    decoder_outputs = layers.LSTM(latent_dim, return_sequences=True)(
        dec_emb, initial_state=[state_h, state_c])
    decoder_outputs = layers.Dense(vocab_size, activation="softmax")(decoder_outputs)

    model = models.Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model


def run_seq2seq(seq_len=6, vocab_size=10):
    X, y = make_seq2seq_data(seq_len=seq_len, vocab_size=vocab_size)
    start_token = vocab_size
    decoder_input = np.hstack([np.full((y.shape[0], 1), start_token), y[:, :-1]])

    X_train, X_test, dec_train, dec_test, y_train, y_test = train_test_split(
        X, decoder_input, y, test_size=0.2, random_state=42
    )

    model = build_seq2seq_model(seq_len, vocab_size + 1)
    model.fit([X_train, dec_train], y_train, validation_split=0.1,
              epochs=40, batch_size=SEQ2SEQ_BATCH, verbose=0)

    pred = np.argmax(model.predict([X_test, dec_test], verbose=0), axis=-1)
    token_acc = (pred == y_test).mean()
    seq_acc = (pred == y_test).all(axis=1).mean()
    print(f"token accuracy: {token_acc:.3f}, sequence accuracy: {seq_acc:.3f}")

    print("sample predictions:")
    for i in range(5):
        print(f"input {X_test[i].tolist()} -> predicted {pred[i].tolist()} "
              f"(expected {y_test[i].tolist()})")

    return model, token_acc, seq_acc


def print_architecture_table(results):
    rows = []
    for r in results:
        rows.append({
            "Model": r["model"],
            "Hidden state": "Yes",
            "Cell state": "Yes" if r["model"] == "LSTM" else "No",
            "Forget gate": "Yes" if r["model"] == "LSTM" else "No",
            "Input gate": "Yes" if r["model"] == "LSTM" else "No",
            "Output gate": "Yes" if r["model"] == "LSTM" else "No",
            "Update gate": "Yes" if r["model"] == "GRU" else "No",
            "Reset gate": "Yes" if r["model"] == "GRU" else "No",
            "Parameters": r["params"],
            "Training time (s)": round(r["train_time"], 2),
            "Test F1 (%)": round(r["f1"], 2),
        })
    df = pd.DataFrame(rows)
    print("\nArchitecture / performance comparison (PDF Section 16):")
    print(df.to_string(index=False))
    return df


# ============================================================
# Main
# ============================================================
if __name__ == "__main__":
    print("=== BPTT numerical exercise ===")
    bptt_numerical_check()

    print("\n=== Part A: HAR sequence classification ===")
    X_train, X_val, X_test, y_train, y_val, y_test = build_har_dataset(subset_size=3000)
    plot_sample_signals(X_train, y_train)

    results = []
    for model_name in ["RNN", "LSTM", "GRU"]:
        print(f"training {model_name}...")
        res = train_and_evaluate(model_name, X_train, y_train, X_val, y_val,
                                 X_test, y_test)
        results.append(res)
        print(f"{model_name}: acc={res['accuracy']:.2f}% f1={res['f1']:.2f}% "
              f"params={res['params']} time={res['train_time']:.1f}s")

    summary = pd.DataFrame([{k: v for k, v in r.items() if k != "confusion"}
                            for r in results])
    print("\nHAR model summary:")
    print(summary.to_string(index=False))
    print_architecture_table(results)

    fig, ax = plt.subplots(figsize=(7, 4))
    pos = np.arange(len(results))
    ax.bar(pos - 0.175, summary["accuracy"], 0.35, label="Accuracy")
    ax.bar(pos + 0.175, summary["f1"], 0.35, label="Macro F1")
    ax.set_xticks(pos)
    ax.set_xticklabels(summary["model"])
    ax.set_ylabel("%")
    ax.set_title("model comparison")
    ax.legend()
    fig.tight_layout()
    fig.savefig(f"{PLOT_DIR}/plot5_model_comparison.png", dpi=150)
    plt.close(fig)

    print("\n=== sequence length experiment ===")
    seq_len_results = sequence_length_experiment(
        X_train, y_train, X_val, y_val, X_test, y_test)
    print(seq_len_results)

    print("\n=== Part B: video understanding (CNN + LSTM) on Kaggle UCF101 ===")
    run_video_pipeline("LSTM")

    print("\n=== Part C: sequence to sequence reversal task ===")
    run_seq2seq()

    print("\nAll plots written to:", PLOT_DIR)

GPUs visible to TensorFlow: 2
=== BPTT numerical exercise ===
h1, h2, h3 = [np.float64(0.3363755443363322), np.float64(0.6163517827505027), np.float64(0.5999579155496113)]

=== Part A: HAR sequence classification ===
Input tensor shape:
Training   : (2470, 128, 9)
Validation : (530, 128, 9)
Testing    : (2947, 128, 9)
Number of classes: 6
Number of features per time step: 9
Sequence length: 128
Class distribution (train): [412 412 412 411 411 412]
training RNN...
RNN: acc=67.80% f1=66.60% params=1974 time=7.6s
training LSTM...
LSTM: acc=84.76% f1=84.59% params=6006 time=6.9s
training GRU...
GRU: acc=82.29% f1=82.29% params=4758 time=6.7s

HAR model summary:
model  accuracy  precision    recall        f1  params  train_time
  RNN 67.797760  66.747814 67.023178 66.600120    1974    7.597087
 LSTM 84.764167  85.079397 84.673587 84.590169    6006    6.934355
  GRU 82.287072  82.472353 82.165018 82.294473    4758    6.655182

Architecture / performance comparison (PDF Section 16):
Model Hid